In [1]:

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score

# -------------------------------------------------------------------------
# 1. Historical Project Data (Synthetic Generation for Zero-Setup Run)
# -------------------------------------------------------------------------
@st.cache_data
def load_historical_data():
    np.random.seed(42)
    n_samples = 1000

    data = {
        'Project_ID': [f'PRJ-{i:04d}' for i in range(1, n_samples + 1)],
        'Area_Acres': np.random.uniform(10, 500, n_samples),
        'Budget_Millions': np.random.uniform(5, 100, n_samples),
        'Number_of_Landowners': np.random.randint(1, 200, n_samples),
        'Zone_Type': np.random.choice(['Urban', 'Semi-Urban', 'Rural'], n_samples),
        'Ecological_Sensitivity': np.random.choice(['Low', 'Medium', 'High'], n_samples),
        'Active_Litigations': np.random.randint(0, 15, n_samples),
        'Public_Resistance_Index': np.random.uniform(0, 10, n_samples),
    }

    df = pd.DataFrame(data)

    # Introduce deliberate missing values for the cleaning step
    df.loc[np.random.choice(df.index, 50), 'Budget_Millions'] = np.nan

    # Target Variable Generation: Delay probability based on features
    delay_prob = (
        (df['Number_of_Landowners'] / 200) * 0.3 +
        (df['Active_Litigations'] / 15) * 0.4 +
        (df['Public_Resistance_Index'] / 10) * 0.3
    )
    df['Delayed'] = (delay_prob + np.random.normal(0, 0.1, n_samples) > 0.5).astype(int)

    return df

# -------------------------------------------------------------------------
# 2. Data Cleaning & 3. Feature Engineering
# -------------------------------------------------------------------------
@st.cache_data
def preprocess_data(df):
    df_clean = df.copy()

    # Data Cleaning: Impute missing budgets with median
    df_clean['Budget_Millions'] = df_clean['Budget_Millions'].fillna(df_clean['Budget_Millions'].median())

    # Feature Engineering: Complexity Score & Density
    df_clean['Landowner_Density'] = df_clean['Number_of_Landowners'] / df_clean['Area_Acres']
    df_clean['Risk_Multiplier'] = df_clean['Active_Litigations'] * df_clean['Public_Resistance_Index']

    # Encoding Categoricals
    le_zone = LabelEncoder()
    le_eco = LabelEncoder()
    df_clean['Zone_Type_Encoded'] = le_zone.fit_transform(df_clean['Zone_Type'])
    df_clean['Eco_Sensitivity_Encoded'] = le_eco.fit_transform(df_clean['Ecological_Sensitivity'])

    features = ['Area_Acres', 'Budget_Millions', 'Number_of_Landowners',
                'Active_Litigations', 'Public_Resistance_Index',
                'Landowner_Density', 'Risk_Multiplier',
                'Zone_Type_Encoded', 'Eco_Sensitivity_Encoded']

    X = df_clean[features]
    y = df_clean['Delayed']

    return df_clean, X, y, features, le_zone, le_eco

# -------------------------------------------------------------------------
# 4. ML Prediction Model Training
# -------------------------------------------------------------------------
@st.cache_resource
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)

    return model, scaler, acc

# -------------------------------------------------------------------------
# 5. Dashboard for Authorities (Streamlit App)
# -------------------------------------------------------------------------
def main():
    st.set_page_config(page_title="SIH26017: Land Acquisition Risk", layout="wide")
    st.title("🚧 Predictive Analytics: Early Detection of Land Acquisition Delays")

    # Load and process data
    raw_df = load_historical_data()
    processed_df, X, y, feature_names, le_zone, le_eco = preprocess_data(raw_df)
    model, scaler, accuracy = train_model(X, y)

    st.sidebar.header("Assess New Project")
    st.sidebar.markdown("Enter project parameters to classify delay risk.")

    # User Inputs
    in_area = st.sidebar.number_input("Area (Acres)", min_value=1.0, value=150.0)
    in_budget = st.sidebar.number_input("Budget (Millions)", min_value=1.0, value=50.0)
    in_owners = st.sidebar.slider("Number of Landowners", 1, 250, 45)
    in_litigations = st.sidebar.slider("Active Litigations in Area", 0, 20, 2)
    in_resistance = st.sidebar.slider("Public Resistance Index (0-10)", 0.0, 10.0, 3.5)
    in_zone = st.sidebar.selectbox("Zone Type", ['Urban', 'Semi-Urban', 'Rural'])
    in_eco = st.sidebar.selectbox("Ecological Sensitivity", ['Low', 'Medium', 'High'])

    # Calculate engineered features for input
    in_density = in_owners / in_area
    in_risk_mult = in_litigations * in_resistance
    in_zone_enc = le_zone.transform([in_zone])[0]
    in_eco_enc = le_eco.transform([in_eco])[0]

    input_data = np.array([[in_area, in_budget, in_owners, in_litigations,
                            in_resistance, in_density, in_risk_mult,
                            in_zone_enc, in_eco_enc]])

    input_scaled = scaler.transform(input_data)

    # Delay Probability & Risk Classification
    prob_delay = model.predict_proba(input_scaled)[0][1] * 100

    st.markdown("### Model Inference")
    col1, col2, col3 = st.columns(3)

    col1.metric("Historical Model Accuracy", f"{accuracy*100:.1f}%")
    col2.metric("Predicted Delay Probability", f"{prob_delay:.1f}%")

    if prob_delay < 30:
        risk_level = "🟢 LOW RISK"
    elif prob_delay < 70:
        risk_level = "🟡 MEDIUM RISK"
    else:
        risk_level = "🔴 HIGH RISK"

    col3.metric("Risk Classification", risk_level)

    st.divider()

    # Exploratory Data Analysis (EDA) + Visualization
    st.markdown("### Exploratory Data Analysis & Feature Importance")

    tab1, tab2 = st.tabs(["Feature Importance", "Historical Delay Distribution"])

    with tab1:
        fig1, ax1 = plt.subplots(figsize=(16, 9), dpi=300)
        importances = model.feature_importances_
        indices = np.argsort(importances)

        ax1.barh(range(len(indices)), importances[indices], color='#4C72B0', align='center')
        ax1.set_yticks(range(len(indices)))
        ax1.set_yticklabels([feature_names[i] for i in indices])
        ax1.set_title("Which Factors Contribute Most to Delays?", fontsize=16, pad=20)
        ax1.set_xlabel("Relative Importance")
        sns.despine()
        st.pyplot(fig1)

    with tab2:
        fig2, ax2 = plt.subplots(figsize=(16, 9), dpi=300)
        sns.kdeplot(data=processed_df, x='Public_Resistance_Index', hue='Delayed',
                    fill=True, common_norm=False, palette=['#55A868', '#C44E52'], ax=ax2)
        ax2.set_title("Public Resistance vs. Project Delays", fontsize=16, pad=20)
        ax2.set_xlabel("Public Resistance Index")
        ax2.set_ylabel("Density")
        sns.despine()
        st.pyplot(fig2)

if __name__ == "__main__":
    main()

2026-08-31 16:44:30.761 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-08-31 16:44:30.764 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-08-31 16:44:30.767 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-31 16:44:30.767 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-31 16:44:31.005 
  command:

    streamlit run /Users/potlapallidiliprao/Library/Python/3.9/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-08-31 16:44:31.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-31 16:44:31.006 Thread 'MainThread': missing ScriptRunContext! This warning c

SyntaxError: invalid syntax (49292858.py, line 1)

In [15]:
!streamlit run dilip.py

Usage: streamlit run [OPTIONS] TARGET [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: dilip.py
